# Tokenización

Reseñas de Amazon (`fancyzhx/amazon_polarity`). El cuaderno sigue los nueve pasos de las diapositivas sobre una reseña, y cierra con los índices del vocabulario. Tiene seis partes:

1. Carga de datos
2. Limpieza inicial
3. Normalizar y separar
4. Ajustar según la tarea
5. Cada palabra recibe un ID
6. La misma tokenización con `get_tokenizer`

## 1. Carga de datos

Cada reseña trae título, texto y etiqueta: 0 es negativa y 1 es positiva. Tomamos las primeras 5000. Seguimos una reseña que trae una etiqueta HTML y una errata, como el ejemplo de clase.

In [ ]:
import re

import torch
from datasets import load_dataset

dataset = load_dataset("fancyzhx/amazon_polarity", split="train", streaming=True)
rows = list(dataset.take(50000))

{'label': 1,
 'title': 'Alaska sourdough',
 'content': 'REad most of the book while visiting my brother in Alaska. Loved it. I am going to be making my sourdough starter soon. Book is full of great stories and recipes.'}

In [10]:
corpus = [row["content"] for row in rows]
text = corpus[42]
text

'REad most of the book while visiting my brother in Alaska. Loved it. I am going to be making my sourdough starter soon. Book is full of great stories and recipes.'

## 2. Limpieza inicial

La salida de cada paso se convierte en la entrada del siguiente. Primero el texto original, después sin etiquetas HTML, y al final en minúsculas.

In [2]:
def lim_inicial(doc):
    sin_html = re.sub(r"<[^>]+>", " ", doc)
    minusculas = " ".join(sin_html.lower().split())
    return minusculas

minusculas = lim_inicial(text)
minusculas

'read most of the book while visiting my brother in alaska. loved it. i am going to be making my sourdough starter soon. book is full of great stories and recipes.'

## 3. Stopwords y corrección de ortografia

Se puede usar un SpellChecker para hacer correcciones. Tambien se pueden quitar palabras funcionales es opcional y depende de la tarea; aquí salen artículos, pronombres y auxiliares.

In [3]:
from spellchecker import SpellChecker

spell = SpellChecker()
def spellchckr(doc):
    corrected_doc = spell.correction(doc)
    return corrected_doc

corregido = spellchckr(minusculas)
corregido


'read most of the book while visiting my brother in alaska. loved it. i am going to be making my sourdough starter soon. book is full of great stories and recipes.'

In [4]:
def stopwords_removal(doc): 
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", doc)
    stopwords = {
        "a", "an", "the", "of", "to", "and", "is", "it", "in", "on", "for", "with",
        "that", "this", "you", "your", "has", "have", "not", "will", "all", "after",
        "one", "plus", "out", "be", "was", "are", "as", "at", "from", "or", "by",
        "its", "i", "my", "we", "they", "their", "so", "if", "but", "do",
    }
    sin_stop = [token for token in tokens if token not in stopwords]
    return sin_stop

sin_stop = stopwords_removal(corregido)
sin_stop

['read',
 'most',
 'book',
 'while',
 'visiting',
 'brother',
 'alaska',
 'loved',
 'am',
 'going',
 'making',
 'sourdough',
 'starter',
 'soon',
 'book',
 'full',
 'great',
 'stories',
 'recipes']

## 4. Ajustar según la tarea

La lematización deja la forma base: `recipes` pasa a `recipe`.

In [5]:
from nltk import WordNetLemmatizer
import nltk
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
def lemmatize(doc):
    lemmatized_doc = [lemmatizer.lemmatize(word) for word in doc]
    return lemmatized_doc

secuencia = lemmatize(sin_stop)
secuencia

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/danielvillarraga/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


['read',
 'most',
 'book',
 'while',
 'visiting',
 'brother',
 'alaska',
 'loved',
 'am',
 'going',
 'making',
 'sourdough',
 'starter',
 'soon',
 'book',
 'full',
 'great',
 'story',
 'recipe']

## 5. Cada palabra recibe un ID

El vocabulario numera los tokens del corpus ya preprocesado, por orden de aparición. El mismo token conserva su índice. La reseña queda como un tensor `long`.

In [6]:
def preprocess_corpus(corpus):
    proccessed_corpus = []
    for doc in corpus:
        try:
            doc = lim_inicial(doc)
            doc = spellchckr(doc)
            doc = stopwords_removal(doc)
            doc = lemmatize(doc)
            proccessed_corpus.append(doc)
        except:
            next
    return proccessed_corpus

pre_corpus = preprocess_corpus(corpus)
print(pre_corpus[42])
print(len(pre_corpus))
    
    

['read', 'most', 'book', 'while', 'visiting', 'brother', 'alaska', 'loved', 'am', 'going', 'making', 'sourdough', 'starter', 'soon', 'book', 'full', 'great', 'story', 'recipe']
49999


In [7]:
vocab = set()
[vocab.update(doc) for doc in pre_corpus]
vocab = sorted(vocab)
len(vocab)

66799

# 6. Naive-Ids

In [8]:
naive_ids = {}
for index, token in enumerate(vocab):
    naive_ids[token] = index
naive_ids

{"a'la": 0,
 "a's": 1,
 'aa': 2,
 "aa's": 3,
 'aaa': 4,
 'aaaa': 5,
 'aaaaaaaaaction': 6,
 'aaaaaaaaarrrggggg': 7,
 'aaah': 8,
 'aaargh': 9,
 'aac': 10,
 'aachen': 11,
 'aadapt': 12,
 'aaf': 13,
 'aafes': 14,
 'aafp': 15,
 "aagard's": 16,
 'aah': 17,
 'aal': 18,
 'aaliyah': 19,
 'aam': 20,
 'aaman': 21,
 'aamt': 22,
 'aan': 23,
 'aand': 24,
 'aap': 25,
 'aare': 26,
 'aargh': 27,
 'aaron': 28,
 "aaron's": 29,
 'aarrgghh': 30,
 'aart': 31,
 'aasimov': 32,
 'aawwww': 33,
 'aax': 34,
 'ab': 35,
 "ab's": 36,
 'aba': 37,
 'aback': 38,
 'abado': 39,
 'abalone': 40,
 'abandon': 41,
 'abandoned': 42,
 'abandoning': 43,
 'abandonment': 44,
 'abanes': 45,
 'abarat': 46,
 'abarca': 47,
 'abated': 48,
 'abb': 49,
 "abb's": 50,
 'abba': 51,
 "abba's": 52,
 'abbado': 53,
 "abbado's": 54,
 'abbath': 55,
 'abbe': 56,
 'abbess': 57,
 'abbey': 58,
 "abbey's": 59,
 'abbility': 60,
 'abble': 61,
 'abbot': 62,
 'abbott': 63,
 'abbreviated': 64,
 'abbreviates': 65,
 'abbreviation': 66,
 'abby': 67,
 'abc': 6

# 7. One-hot Encoding

In [20]:

doc = pre_corpus[42]
one_hot_sequence = []
for token in doc:
    one_hot = torch.zeros(len(vocab))
    one_hot[naive_ids[token]] = 1
    one_hot_sequence.append(one_hot)
one_hot_sequence

[tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.]),
 tensor([0., 0., 0.,  ..., 0., 0., 0.])]